In [ ]:
import torch
from torch_geometric.data import Data
from torch_geometric.loader import NeighborLoader
from torch_geometric.datasets import Planetoid
from torch_geometric.transforms import NormalizeFeatures
from sklearn.metrics import accuracy_score, roc_auc_score

# Load the dataset
dataset = Planetoid(root='/tmp/Cora', name='Cora', transform=NormalizeFeatures())

# Get the first graph in the dataset
data = dataset[0]

loader = NeighborLoader(
    data,
    input_nodes=torch.tensor([0, 1]),
    num_neighbors=[2, 1],
    batch_size=1,
    replace=False,
    shuffle=False,
)

In [25]:
from torch_geometric.nn import GraphSAGE

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = GraphSAGE(
    in_channels=dataset.num_node_features,
    hidden_channels=64,
    out_channels=dataset.num_classes,
    num_layers=2
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [ ]:
import torch
import torch.nn.functional as F
from tqdm import tqdm # Standard for progress bars

def train_model(model, loader, optimizer, device):
    model.train()
    total_loss = 0
    
    progress_bar = tqdm(loader, desc="Training", unit="batch")
    
    for batch in progress_bar:
        print([batch.n_id])
        optimizer.zero_grad()
        batch = batch.to(device)
        
        out = model(batch.x, batch.edge_index)

        train_out = out
        train_y = batch.y

        loss = F.cross_entropy(train_out, train_y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        progress_bar.set_postfix({"loss": f"{loss.item():.4f}", "acc": f"{(train_out.argmax(dim=1) == train_y).sum().item() / train_y.size(0):.4f}"})

    return total_loss / len(loader)

def test_model(model, loader, device):
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            out = model(batch.x, batch.edge_index)
            
            test_out = out
            test_y = batch.y
            
            pred = test_out.argmax(dim=1)
            correct += (pred == test_y).sum().item()
            total += test_y.size(0)
            
    return correct / total


for epoch in range(1, 11):
    loss = train_model(model, loader, optimizer, device)
    acc = test_model(model, loader, device)
    
    # Print overall epoch progress
    print(f"Epoch: {epoch:03d} | Loss: {loss:.4f} | Test Acc: {acc:.4f}")

Training: 100%|██████████| 2/2 [00:00<00:00, 36.90batch/s, loss=1.8891]


[tensor([   0,  633, 2582])]
[tensor([  1,   2, 654, 332])]
Epoch: 001 | Loss: 1.9085 | Test Acc: 0.6250


Training: 100%|██████████| 2/2 [00:00<00:00, 60.55batch/s, loss=1.7904]


[tensor([   0,  633, 2582, 1701])]
[tensor([   1,    2,  654, 1454])]
Epoch: 002 | Loss: 1.7744 | Test Acc: 0.5000


Training: 100%|██████████| 2/2 [00:00<00:00, 61.20batch/s, loss=1.6688]


...

[tensor([   0,  633, 1862, 1701])]
[tensor([  1,   2, 652, 332])]
Epoch: 099 | Loss: 0.0023 | Test Acc: 1.0000


Training:   0%|          | 0/2 [00:00<?, ?batch/s]

[tensor([   0,  633, 2582, 1701])]


Training: 100%|██████████| 2/2 [00:00<00:00, 51.44batch/s, loss=0.0038]

[tensor([  1,   2, 652, 332])]
Epoch: 100 | Loss: 0.0023 | Test Acc: 1.0000
